In [ ]:
%pip install pandas numpy scikit-learn xgboost matplotlib seaborn statsmodels joblib torch

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Load and prepare data
df = pd.read_csv('/content/yt_dataset_v6.csv', header=0)

# Define preprocessing steps
def preprocess_data(data):
    df_processed = data.copy()
    # Handle missing values
    df_processed['log_comment_count'] = df_processed['log_comment_count'].fillna(df_processed['log_comment_count'].median())
    df_processed['log_likes'] = df_processed['log_likes'].fillna(df_processed['log_likes'].median())
    return df_processed

# Define features
exclude_cols = ['video_id', 'channel_id', 'published_at', 'dislikes', 'log_dislikes', 'percentage_dislikes', 'log_percentage_dislikes']
feature_cols = [col for col in df.columns if col not in exclude_cols] #and not col.startswith('log')
X = df[feature_cols]
y = df['log_dislikes']

# Create a preprocessing pipeline
preprocessing_pipeline = Pipeline([
    ('preprocess', FunctionTransformer(func=preprocess_data, validate=False))
])

In [ ]:
# 70% train, 15% validation, 15% test - Chronological split (no shuffle)
# Convert y to numeric, coercing errors to NaN
y = pd.to_numeric(y, errors='coerce')
# Fill any NaN values that resulted from coercion, e.g., with the median
y = y.fillna(y.median())

# First split: 15% test at the end of the timeline
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, shuffle=False
)

# Second split: 15% validation from the remaining 85% (~17.6% of temp)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1764, shuffle=False
)

print(f"Train shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import torch

# Enhanced XGBoost with comprehensive hyperparameter tuning
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# Check for CUDA availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'

xgb_model = xgb.XGBRegressor(
    # objective='reg:absoluteerror',
    objective='reg:squarederror',
    random_state=42,
    tree_method='hist',
    device=device
)

# Use a PredefinedSplit instead of KFold for the 70-15-15 strategy
from sklearn.model_selection import PredefinedSplit

# Create a validation set for RandomizedSearchCV
# -1 for training, 0 for validation
test_fold = np.concatenate([-1 * np.ones(len(X_train)), np.zeros(len(X_val))])
ps = PredefinedSplit(test_fold)

# Combine train and val for the search
X_combined = pd.concat([X_train, X_val])
y_combined = pd.concat([y_train, y_val])

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=100,
    cv=ps, # Using the non-random predefined validation split
    # scoring='neg_mean_absolute_error',
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

random_search.fit(X_combined, y_combined)
best_model = random_search.best_estimator_

In [ ]:
print("Best parameters found by RandomizedSearchCV:")
print(random_search.best_params_)

In [ ]:
# Predictions on all sets
y_train_pred = best_model.predict(X_train)
y_val_pred = best_model.predict(X_val)
y_test_pred = best_model.predict(X_test)

# Evaluation metrics
def evaluate_model(y_true, y_pred, dataset_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"{dataset_name} - RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2}

train_metrics = evaluate_model(y_train, y_train_pred, "Train")
val_metrics = evaluate_model(y_val, y_val_pred, "Validation")
test_metrics = evaluate_model(y_test, y_test_pred, "Test")

In [ ]:
import os
plt.figure(figsize=(12, 8))
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()

output_dir = '/content/output' # Define output directory
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist
plt.savefig(os.path.join(output_dir, 'feature_importance.png')) # Save to output directory
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets = [(y_train, y_train_pred, 'Train'),
           (y_val, y_val_pred, 'Validation'),
           (y_test, y_test_pred, 'Test')]

for i, (y_true, y_pred, title) in enumerate(datasets):
    axes[i].scatter(y_true, y_pred, alpha=0.5)
    axes[i].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    axes[i].set_xlabel('Actual Log Dislikes')
    axes[i].set_ylabel('Predicted Log Dislikes')
    axes[i].set_title(f'{title} Set\nR² = {r2_score(y_true, y_pred):.4f}')

plt.tight_layout()

output_dir = '/content/output' # Define output directory
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist
plt.savefig(os.path.join(output_dir, 'actual_vs_predicted.png')) # Save to output directory
plt.show()

In [ ]:
residuals = y_test - y_test_pred
plt.figure(figsize=(10, 6))
plt.scatter(y_test_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Log Dislikes')
plt.ylabel('Residuals')
plt.title('Residual Plot (Test Set)')

output_dir = '/content/output' # Define output directory
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist
plt.savefig(os.path.join(output_dir, 'residual_plot.png')) # Save to output directory
plt.show()


# Residual distribution
plt.figure(figsize=(10, 6))
plt.hist(residuals, bins=50, alpha=0.7)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Residual Distribution (Test Set)')

output_dir = '/content/output' # Define output directory
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist
plt.savefig(os.path.join(output_dir, 'residual_distribution.png')) # Save to output directory
plt.show()

In [ ]:
def plot_tail_performance(y_true, y_pred, dataset_name, tail_percent=10):
    """Generates scatter plots for the lowest and highest tails of the data."""
    df_temp = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})
    df_temp = df_temp.sort_values(by='y_true')

    tail_size = int(len(df_temp) * tail_percent / 100)

    output_dir = '/content/output' # Define output directory
    os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist


    # Lowest tail plot
    lowest_tail_df = df_temp.head(tail_size)
    plt.figure(figsize=(8, 6))
    plt.scatter(lowest_tail_df['y_true'], lowest_tail_df['y_pred'], alpha=0.5)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    plt.xlabel('Actual Log Dislikes')
    plt.ylabel('Predicted Log Dislikes')
    plt.title(f'{dataset_name} - Lowest {tail_percent}% Tail Performance')
    plt.savefig(os.path.join(output_dir, f'{dataset_name}_lowest_tail_plot.png')) # Save to output directory
    plt.show()

    # Highest tail plot
    highest_tail_df = df_temp.tail(tail_size)
    plt.figure(figsize=(8, 6))
    plt.scatter(highest_tail_df['y_true'], highest_tail_df['y_pred'], alpha=0.5)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    plt.xlabel('Actual Log Dislikes')
    plt.ylabel('Predicted Log Dislikes')
    plt.title(f'{dataset_name} - Highest {tail_percent}% Tail Performance')
    plt.savefig(os.path.join(output_dir, f'{dataset_name}_highest_tail_plot.png')) # Save to output directory
    plt.show()

# Generate tail performance plots for each dataset
plot_tail_performance(y_train, y_train_pred, "Train")
plot_tail_performance(y_val, y_val_pred, "Validation")
plot_tail_performance(y_test, y_test_pred, "Test")

In [ ]:
def analyze_tail_performance(y_true, y_pred, dataset_name, tail_percent=10):
    """Analyzes model performance on the lowest and highest tails of the data."""
    df_temp = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})
    df_temp = df_temp.sort_values(by='y_true')

    # Calculate tail sizes
    tail_size = int(len(df_temp) * tail_percent / 100)

    # Lowest tail
    lowest_tail_df = df_temp.head(tail_size)
    print(f"\n{dataset_name} - Lowest {tail_percent}% Tail Performance:")
    evaluate_model(lowest_tail_df['y_true'], lowest_tail_df['y_pred'], f"{dataset_name} Lowest Tail")

    # Highest tail
    highest_tail_df = df_temp.tail(tail_size)
    print(f"\n{dataset_name} - Highest {tail_percent}% Tail Performance:")
    evaluate_model(highest_tail_df['y_true'], highest_tail_df['y_pred'], f"{dataset_name} Highest Tail")

# Analyze tail performance for each dataset
analyze_tail_performance(y_train, y_train_pred, "Train")
analyze_tail_performance(y_val, y_val_pred, "Validation")
analyze_tail_performance(y_test, y_test_pred, "Test")

In [ ]:
residuals = y_test - y_test_pred

# Create a temporary DataFrame to sort by actual values
df_temp = pd.DataFrame({'y_true': y_test, 'y_pred': y_test_pred, 'residuals': residuals})
df_temp = df_temp.sort_values(by='y_true')

# Calculate tail size
tail_percent = 10
tail_size = int(len(df_temp) * tail_percent / 100)

# Get lowest and highest tail data
lowest_tail_df = df_temp.head(tail_size)
highest_tail_df = df_temp.tail(tail_size)

# Combine tail data
tail_residuals_df = pd.concat([lowest_tail_df, highest_tail_df])

output_dir = '/content/output' # Define output directory
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist

plt.figure(figsize=(10, 6))
plt.scatter(tail_residuals_df['y_pred'], tail_residuals_df['residuals'], alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Log Dislikes')
plt.ylabel('Residuals')
plt.title('Residual Plot (Top and Bottom 10% Tails)')
plt.savefig(os.path.join(output_dir, 'residual_plot_tails.png')) # Save to output directory
plt.show()

# Residual distribution for tails
plt.figure(figsize=(10, 6))
plt.hist(tail_residuals_df['residuals'], bins=50, alpha=0.7)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Residual Distribution (Top and Bottom 10% Tails)')
plt.savefig(os.path.join(output_dir, 'residual_distribution_tails.png')) # Save to output directory
plt.show()

In [ ]:
import statsmodels.api as sm

# Calculate residuals for each set
residuals_train = y_train - y_train_pred
residuals_val = y_val - y_val_pred
residuals_test = y_test - y_test_pred

# Create Q-Q plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sm.qqplot(residuals_train, line='s', ax=axes[0])
axes[0].set_title('Q-Q Plot (Train Set Residuals)')

sm.qqplot(residuals_val, line='s', ax=axes[1])
axes[1].set_title('Q-Q Plot (Validation Set Residuals)')

sm.qqplot(residuals_test, line='s', ax=axes[2])
axes[2].set_title('Q-Q Plot (Test Set Residuals)')

plt.tight_layout()

output_dir = '/content/output' # Define output directory
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist
plt.savefig(os.path.join(output_dir, 'qq_plots.png')) # Save to output directory
plt.show()

In [ ]:
import joblib
from sklearn.pipeline import Pipeline

# Create the full pipeline
full_pipeline = Pipeline([
    ('preprocessing', preprocessing_pipeline),
    ('model', best_model)
])

# Define output directory
output_dir = '/content/output'
os.makedirs(output_dir, exist_ok=True) # Create directory if it doesn't exist

# Save the full pipeline to the output directory
joblib.dump(full_pipeline, os.path.join(output_dir, 'full_xgboost_pipeline.pkl'))

print("Full pipeline saved as full_xgboost_pipeline.pkl")

In [ ]:
import os
import shutil

source_dir = '/content/output'
destination_dir = '/content/drive/My Drive/YT_Dislikes/final_xgboost'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# List all files in the source directory
files_in_source = os.listdir(source_dir)

# Filter for CSV files and copy them
for file_name in files_in_source:
    if not file_name.startswith('.'):
        source_path = os.path.join(source_dir, file_name)
        destination_path = os.path.join(destination_dir, file_name)
        try:
            # Use shutil.copy2 for copying with metadata
            shutil.copy2(source_path, destination_path)
            print(f"Copied '{file_name}' to '{destination_dir}'")
        except FileNotFoundError:
            print(f"Error: File '{file_name}' not found.")
        except Exception as e:
            print(f"Error copying file '{file_name}': {e}")

print("CSV file copying process completed.")

In [ ]:
import joblib
import pandas as pd
import os

# Define the path to the saved pipeline
output_dir = '/content/output'
pipeline_path = os.path.join(output_dir, 'full_xgboost_pipeline.pkl')

# Load the pipeline
loaded_pipeline = joblib.load(pipeline_path)
print("Pipeline loaded successfully.")

In [ ]:
# Create sample data with the same features as the training data
# You should replace this with your actual new data
sample_data = {
    "age": [287],
    "avg_compound": [0.042724],
    "avg_neg": [0.03814],
    "avg_neu": [0.89316],
    "avg_pos": [0.0687],
    "comment_count": [2086],
    "comment_sample_size": [50],
    "likes": [23342],
    "view_count": [1355784],
    "duration": [397],
    "genre_id": [13],
    "log_comment_count": [3.319314304],
    "log_likes": [10.05805243],
    "log_view_count": [14.11989118],
    "view_like_ratio": [58.08096646],
    "log_view_like_ratio": [4.078908816]
}

sample_df = pd.DataFrame(sample_data)

# Get the feature names used during training from the global feature_cols variable
# Ensure the sample_df columns are in the same order as feature_cols
# Also, ensure only relevant features are included if feature_cols was updated elsewhere.
# For this notebook, feature_cols is defined in cell vW5I8G_JPTxo.
if 'feature_cols' in globals():
    sample_df = sample_df[feature_cols]
else:
    print("Warning: feature_cols not found. Proceeding with sample_df as is, may lead to errors if column order mismatches.")

# Predict dislikes using the loaded pipeline
predicted_log_dislikes = loaded_pipeline.predict(sample_df)

# Inverse transform the log predictions to get the actual predicted dislikes
predicted_dislikes = np.expm1(predicted_log_dislikes)

print("\nSample Data:")
display(sample_df)

print("\nPredicted Log Dislikes:")
print(predicted_log_dislikes)

print("\nPredicted Dislikes:")
print(predicted_dislikes)